In [2]:
import sqlite3
import pandas as pd
from datetime import datetime

In [3]:
student_data = [
    (1, "Nguyen Minh Hoang", "May Tinh", 12, 6.7),
    (2, "Tran Thi Lan", "Kinh Te", 34, 9.2),
    (3, "Pham Van Nam", "Toan Tin", None, 7.9),
    (4, "Le Thanh Huyen", "Toan Tin", 20, 7.2),
    (5, "Vu Quoc Anh", "May Tinh", 24, 8.0),
    (6, "Dang Thuy Linh", "May Tinh", 24, 5.5),
    (7, "Bui Tien Dung", "Kinh Te", 34, 9.2),
    (8, "Ho Ngoc Mai", "Toan Tin", 20, 8.8),
    (9, "Duong Huu Phuc", "Kinh Te", None, 7.2),
    (10, "Cao Thi Hanh", "May Tinh", None, 7.0)
]

course_data = [
    (12, "Giai tich"),
    (34, "Thong ke"),
    (26, "Tin hoc")
]

In [4]:
conn = sqlite3.connect("baitap_chuong2.db")
cursor = conn.cursor()

In [5]:
cursor.execute("DROP TABLE IF EXISTS student")
cursor.execute("DROP TABLE IF EXISTS course")
conn.commit()

In [6]:
cursor.execute("""
CREATE TABLE student (
    student_id INTEGER,
    name TEXT,
    class TEXT,
    course_id INTEGER,
    score REAL
)
""")

cursor.execute("""
CREATE TABLE course (
    id INTEGER,
    course_name TEXT
)
""")

conn.commit()

In [7]:
cursor.executemany("INSERT INTO student VALUES (?, ?, ?, ?, ?)", student_data)
cursor.executemany("INSERT INTO course VALUES (?, ?)", course_data)
conn.commit()

In [8]:
print("Bảng student:")
display(pd.read_sql_query("SELECT * FROM student", conn))

print("Bảng course:")
display(pd.read_sql_query("SELECT * FROM course", conn))

Bảng student:


,student_id,name,class,course_id,score
0,1,Nguyen Minh Hoang,May Tinh,12.0,6.7
1,2,Tran Thi Lan,Kinh Te,34.0,9.2
2,3,Pham Van Nam,Toan Tin,NaN,7.9
3,4,Le Thanh Huyen,Toan Tin,20.0,7.2
4,5,Vu Quoc Anh,May Tinh,24.0,8.0
5,6,Dang Thuy Linh,May Tinh,24.0,5.5
6,7,Bui Tien Dung,Kinh Te,34.0,9.2
7,8,Ho Ngoc Mai,Toan Tin,20.0,8.8
8,9,Duong Huu Phuc,Kinh Te,NaN,7.2
9,10,Cao Thi Hanh,May Tinh,NaN,7.0


Bảng course:


,id,course_name
0,12,Giai tich
1,34,Thong ke
2,26,Tin hoc


## Sử dụng tích DESCARTES


In [9]:
query = """
SELECT *
FROM student, course
"""
df = pd.read_sql_query(query, conn)
display(df)

,student_id,name,class,course_id,score,id,course_name
0,1,Nguyen Minh Hoang,May Tinh,12.0,6.7,12,Giai tich
1,1,Nguyen Minh Hoang,May Tinh,12.0,6.7,34,Thong ke
2,1,Nguyen Minh Hoang,May Tinh,12.0,6.7,26,Tin hoc
3,2,Tran Thi Lan,Kinh Te,34.0,9.2,12,Giai tich
4,2,Tran Thi Lan,Kinh Te,34.0,9.2,34,Thong ke
5,2,Tran Thi Lan,Kinh Te,34.0,9.2,26,Tin hoc
6,3,Pham Van Nam,Toan Tin,NaN,7.9,12,Giai tich
7,3,Pham Van Nam,Toan Tin,NaN,7.9,34,Thong ke
8,3,Pham Van Nam,Toan Tin,NaN,7.9,26,Tin hoc
9,4,Le Thanh Huyen,Toan Tin,20.0,7.2,12,Giai tich


### Nhận xét:
Tích Descartes ghép mỗi sinh viên với tất cả môn học, nên số dòng kết quả rất lớn và chưa phản ánh đúng quan hệ thực tế.


## Sử dụng INNER JOIN

In [10]:

query = """
SELECT s.*, c.course_name
FROM student s
INNER JOIN course c
ON s.course_id = c.id
"""
df = pd.read_sql_query(query, conn)
display(df)

,student_id,name,class,course_id,score,course_name
0,1,Nguyen Minh Hoang,May Tinh,12,6.7,Giai tich
1,2,Tran Thi Lan,Kinh Te,34,9.2,Thong ke
2,7,Bui Tien Dung,Kinh Te,34,9.2,Thong ke


### Nhận xét:
INNER JOIN chỉ lấy những sinh viên có `course_id` khớp với bảng `course`.

## Sử dụng LEFT JOIN

In [11]:

query = """
SELECT s.*, c.course_name
FROM student s
LEFT JOIN course c
ON s.course_id = c.id
"""
df = pd.read_sql_query(query, conn)
display(df)

,student_id,name,class,course_id,score,course_name
0,1,Nguyen Minh Hoang,May Tinh,12.0,6.7,Giai tich
1,2,Tran Thi Lan,Kinh Te,34.0,9.2,Thong ke
2,3,Pham Van Nam,Toan Tin,NaN,7.9,None
3,4,Le Thanh Huyen,Toan Tin,20.0,7.2,None
4,5,Vu Quoc Anh,May Tinh,24.0,8.0,None
5,6,Dang Thuy Linh,May Tinh,24.0,5.5,None
6,7,Bui Tien Dung,Kinh Te,34.0,9.2,Thong ke
7,8,Ho Ngoc Mai,Toan Tin,20.0,8.8,None
8,9,Duong Huu Phuc,Kinh Te,NaN,7.2,None
9,10,Cao Thi Hanh,May Tinh,NaN,7.0,None


### Nhận xét:
LEFT JOIN giữ toàn bộ sinh viên, kể cả những bạn chưa có hoặc có mã môn không hợp lệ.

## Sử dụng RIGHT JOIN

In [12]:

query = """
SELECT s.*, c.course_name
FROM course c
LEFT JOIN student s
ON s.course_id = c.id
"""
df = pd.read_sql_query(query, conn)
display(df)

,student_id,name,class,course_id,score,course_name
0,1.0,Nguyen Minh Hoang,May Tinh,12.0,6.7,Giai tich
1,2.0,Tran Thi Lan,Kinh Te,34.0,9.2,Thong ke
2,7.0,Bui Tien Dung,Kinh Te,34.0,9.2,Thong ke
3,NaN,None,None,NaN,NaN,Tin hoc


### Nhận xét:
SQLite không hỗ trợ RIGHT JOIN trực tiếp trong nhiều phiên bản, nên ta mô phỏng bằng cách đảo thứ tự bảng và dùng LEFT JOIN.

## Sử dụng FULL OUTER JOIN

In [13]:
#FULL OUTER JOIN
query = """
SELECT s.student_id, s.name, s.class, s.course_id, s.score, c.course_name
FROM student s
LEFT JOIN course c
ON s.course_id = c.id

UNION

SELECT s.student_id, s.name, s.class, s.course_id, s.score, c.course_name
FROM course c
LEFT JOIN student s
ON s.course_id = c.id
"""
df = pd.read_sql_query(query, conn)
display(df)

,student_id,name,class,course_id,score,course_name
0,NaN,None,None,NaN,NaN,Tin hoc
1,1.0,Nguyen Minh Hoang,May Tinh,12.0,6.7,Giai tich
2,2.0,Tran Thi Lan,Kinh Te,34.0,9.2,Thong ke
3,3.0,Pham Van Nam,Toan Tin,NaN,7.9,None
4,4.0,Le Thanh Huyen,Toan Tin,20.0,7.2,None
5,5.0,Vu Quoc Anh,May Tinh,24.0,8.0,None
6,6.0,Dang Thuy Linh,May Tinh,24.0,5.5,None
7,7.0,Bui Tien Dung,Kinh Te,34.0,9.2,Thong ke
8,8.0,Ho Ngoc Mai,Toan Tin,20.0,8.8,None
9,9.0,Duong Huu Phuc,Kinh Te,NaN,7.2,None


### Nhận xét:
FULL OUTER JOIN giúp lấy toàn bộ dữ liệu của cả hai bảng, kể cả phần không khớp.

## CÂU 2.1: CẬP NHẬT course_id CÒN THIẾU

In [14]:
# Cập nhật giá trị course_id còn thiếu thành 26 (Tin hoc)
cursor.execute("""
UPDATE student
SET course_id = 26
WHERE course_id IS NULL
""")
conn.commit()

print("Sau khi cập nhật course_id bị thiếu:")
display(pd.read_sql_query("SELECT * FROM student", conn))

Sau khi cập nhật course_id bị thiếu:


,student_id,name,class,course_id,score
0,1,Nguyen Minh Hoang,May Tinh,12,6.7
1,2,Tran Thi Lan,Kinh Te,34,9.2
2,3,Pham Van Nam,Toan Tin,26,7.9
3,4,Le Thanh Huyen,Toan Tin,20,7.2
4,5,Vu Quoc Anh,May Tinh,24,8.0
5,6,Dang Thuy Linh,May Tinh,24,5.5
6,7,Bui Tien Dung,Kinh Te,34,9.2
7,8,Ho Ngoc Mai,Toan Tin,20,8.8
8,9,Duong Huu Phuc,Kinh Te,26,7.2
9,10,Cao Thi Hanh,May Tinh,26,7.0


## CÂU 2.2: XÓA DỮ LIỆU KHÔNG HỢP LỆ

In [15]:
cursor.execute("""
DELETE FROM student
WHERE course_id NOT IN (SELECT id FROM course)
""")
conn.commit()

print("Sau khi xóa các bản ghi có course_id không tồn tại:")
display(pd.read_sql_query("SELECT * FROM student", conn))

Sau khi xóa các bản ghi có course_id không tồn tại:


,student_id,name,class,course_id,score
0,1,Nguyen Minh Hoang,May Tinh,12,6.7
1,2,Tran Thi Lan,Kinh Te,34,9.2
2,3,Pham Van Nam,Toan Tin,26,7.9
3,7,Bui Tien Dung,Kinh Te,34,9.2
4,9,Duong Huu Phuc,Kinh Te,26,7.2
5,10,Cao Thi Hanh,May Tinh,26,7.0


Nhận xét:
Đã bổ sung dữ liệu thiếu
đã loại bỏ các course_id không tồn tại

## 2a. THỐNG KÊ THEO LỚP

In [16]:
df = pd.read_sql("""
SELECT class,
       COUNT(*) AS tong_sv,
       ROUND(AVG(score),2) AS diem_tb
FROM student
GROUP BY class
""", conn)

display(df)

,class,tong_sv,diem_tb
0,Kinh Te,3,8.53
1,May Tinh,2,6.85
2,Toan Tin,1,7.90


Kết luận:
Biết được số lượng sinh viên từng lớp
so sánh chất lượng học tập giữa các lớp

## 2b. THỐNG KÊ THEO MÔN

In [17]:
df = pd.read_sql("""
SELECT c.course_name,
       COUNT(*) AS tong_sv,
       ROUND(AVG(s.score),2) AS diem_tb
FROM student s
JOIN course c ON s.course_id = c.id
GROUP BY c.course_name
""", conn)

display(df)

,course_name,tong_sv,diem_tb
0,Giai tich,1,6.70
1,Thong ke,2,9.20
2,Tin hoc,3,7.37


Kết Luận : Biết được môn nào tốt môn nào kém

## 2c. PHÂN LOẠI

In [18]:
df = pd.read_sql("""
SELECT c.course_name,
       ROUND(AVG(s.score),2) AS diem_tb,
       CASE
           WHEN AVG(s.score) >= 9 THEN 'Xuat sac'
           WHEN AVG(s.score) >= 6 THEN 'Tot'
           ELSE 'Kem'
       END AS xep_loai
FROM student s
JOIN course c ON s.course_id = c.id
GROUP BY c.course_name
""", conn)

display(df)

,course_name,diem_tb,xep_loai
0,Giai tich,6.70,Tot
1,Thong ke,9.20,Xuat sac
2,Tin hoc,7.37,Tot


Giúp đánh giá chất lượng từng môn:

≥ 9 → Xuất sắc

6–8.9 → Tốt

< 6 → Kém

## 3a.Xếp hạng sinh viên thông qua điểm số.

Xếp hạng toàn bộ

In [19]:
df = pd.read_sql("""
SELECT *,
RANK() OVER (ORDER BY score DESC) AS rank
FROM student
""", conn)

display(df)

,student_id,name,class,course_id,score,rank
0,2,Tran Thi Lan,Kinh Te,34,9.2,1
1,7,Bui Tien Dung,Kinh Te,34,9.2,1
2,3,Pham Van Nam,Toan Tin,26,7.9,3
3,9,Duong Huu Phuc,Kinh Te,26,7.2,4
4,10,Cao Thi Hanh,May Tinh,26,7.0,5
5,1,Nguyen Minh Hoang,May Tinh,12,6.7,6


Nhận xét:
- Sinh viên được xếp hạng theo điểm từ cao xuống thấp
- Không phân biệt lớp hay môn

Top 3 sinh viên cao điểm nhất

In [21]:
df = pd.read_sql("""
SELECT * FROM (
SELECT *,
RANK() OVER (ORDER BY score DESC) AS r
FROM student)
WHERE r <= 3
""", conn)

display(df)

,student_id,name,class,course_id,score,r
0,2,Tran Thi Lan,Kinh Te,34,9.2,1
1,7,Bui Tien Dung,Kinh Te,34,9.2,1
2,3,Pham Van Nam,Toan Tin,26,7.9,3


Nhận xét:
- Xác định 3 bạn điểm tốt nhất

Top 3 sinh viên thấp điểm nhất


In [22]:
df = pd.read_sql("""
SELECT * FROM (
SELECT *,
RANK() OVER (ORDER BY score ASC) AS r
FROM student)
WHERE r <= 3
""", conn)

display(df)

,student_id,name,class,course_id,score,r
0,1,Nguyen Minh Hoang,May Tinh,12,6.7,1
1,10,Cao Thi Hanh,May Tinh,26,7.0,2
2,9,Duong Huu Phuc,Kinh Te,26,7.2,3


Nhận xét:

- Top 3 sinh viên điểm thấp nhất

## 3b. XẾP HẠNG THEO LỚP

In [32]:
df = pd.read_sql("""
SELECT *,
       ROW_NUMBER() OVER (PARTITION BY class ORDER BY score DESC) AS rank_in_class
FROM student
""", conn)

display(df)

,student_id,name,class,course_id,score,rank_in_class
0,2,Tran Thi Lan,Kinh Te,34,9.2,1
1,7,Bui Tien Dung,Kinh Te,34,9.2,2
2,9,Duong Huu Phuc,Kinh Te,26,7.2,3
3,10,Cao Thi Hanh,May Tinh,26,7.0,1
4,1,Nguyen Minh Hoang,May Tinh,12,6.7,2
5,3,Pham Van Nam,Toan Tin,26,7.9,1


Nhận xét:

- Sinh viên được xếp hạng riêng trong từng lớp
- Điểm cao hơn thì hạng cao hơn
- Nếu bằng điểm thì có thể cùng hạng

3b.2 Top 3 sinh viên cao nhất theo từng lớp

In [33]:
classes = pd.read_sql("SELECT DISTINCT class FROM student", conn)

for lop in classes["class"]:
    print(f"\nTOP 3 cao nhất lớp: {lop}")

    df = pd.read_sql(f"""
    SELECT *
    FROM (
        SELECT *,
               ROW_NUMBER() OVER (PARTITION BY class ORDER BY score DESC) AS rank_in_class
        FROM student
    )
    WHERE class = '{lop}' AND rank_in_class <= 3
    ORDER BY rank_in_class
    """, conn)

    display(df)


TOP 3 cao nhất lớp: May Tinh


,student_id,name,class,course_id,score,rank_in_class
0,10,Cao Thi Hanh,May Tinh,26,7.0,1
1,1,Nguyen Minh Hoang,May Tinh,12,6.7,2



TOP 3 cao nhất lớp: Kinh Te


,student_id,name,class,course_id,score,rank_in_class
0,2,Tran Thi Lan,Kinh Te,34,9.2,1
1,7,Bui Tien Dung,Kinh Te,34,9.2,2
2,9,Duong Huu Phuc,Kinh Te,26,7.2,3



TOP 3 cao nhất lớp: Toan Tin


,student_id,name,class,course_id,score,rank_in_class
0,3,Pham Van Nam,Toan Tin,26,7.9,1


Kết luận:

- Truy vấn này cho biết 3 sinh viên có điểm cao nhất trong mỗi lớp

- Giúp nhận diện những sinh viên có kết
quả học tập tốt trong lớp

3b.3 Top 3 sinh viên thấp nhất theo từng lớp

In [34]:
classes = pd.read_sql("SELECT DISTINCT class FROM student", conn)

for lop in classes["class"]:
    print(f"\nTOP 3 thấp nhất lớp: {lop}")

    df = pd.read_sql(f"""
    SELECT *
    FROM (
        SELECT *,
               ROW_NUMBER() OVER (PARTITION BY class ORDER BY score ASC) AS rank_in_class
        FROM student
    )
    WHERE class = '{lop}' AND rank_in_class <= 3
    ORDER BY rank_in_class
    """, conn)

    display(df)


TOP 3 thấp nhất lớp: May Tinh


,student_id,name,class,course_id,score,rank_in_class
0,1,Nguyen Minh Hoang,May Tinh,12,6.7,1
1,10,Cao Thi Hanh,May Tinh,26,7.0,2



TOP 3 thấp nhất lớp: Kinh Te


,student_id,name,class,course_id,score,rank_in_class
0,9,Duong Huu Phuc,Kinh Te,26,7.2,1
1,2,Tran Thi Lan,Kinh Te,34,9.2,2
2,7,Bui Tien Dung,Kinh Te,34,9.2,3



TOP 3 thấp nhất lớp: Toan Tin


,student_id,name,class,course_id,score,rank_in_class
0,3,Pham Van Nam,Toan Tin,26,7.9,1


Kết luận:
- Mỗi lớp có danh sách riêng giúp tìm ra những người điểm thấp nhất
- Giúp đánh giá và cải thiện kết quả với những bạn điểm thấp

## 3c. XẾP HẠNG THEO MÃ MÔN HỌC

In [35]:
df = pd.read_sql("""
SELECT *,
       ROW_NUMBER() OVER (PARTITION BY course_id ORDER BY score DESC) AS rank_in_course
FROM student
""", conn)

display(df)

,student_id,name,class,course_id,score,rank_in_course
0,1,Nguyen Minh Hoang,May Tinh,12,6.7,1
1,3,Pham Van Nam,Toan Tin,26,7.9,1
2,9,Duong Huu Phuc,Kinh Te,26,7.2,2
3,10,Cao Thi Hanh,May Tinh,26,7.0,3
4,2,Tran Thi Lan,Kinh Te,34,9.2,1
5,7,Bui Tien Dung,Kinh Te,34,9.2,2


Nhận xét:
- Sinh viên được xếp hạng theo từng môn học
- So sánh trong cùng một môn

3c.2 Top 3 sinh viên cao nhất theo từng môn

In [36]:
courses = pd.read_sql("SELECT DISTINCT course_id FROM student", conn)

for mon in courses["course_id"]:
    print(f"\nTOP 3 cao nhất môn: {mon}")

    df = pd.read_sql(f"""
    SELECT *
    FROM (
        SELECT *,
               ROW_NUMBER() OVER (PARTITION BY course_id ORDER BY score DESC) AS rank_in_course
        FROM student
    )
    WHERE course_id = {mon} AND rank_in_course <= 3
    ORDER BY rank_in_course
    """, conn)

    display(df)


TOP 3 cao nhất môn: 12


,student_id,name,class,course_id,score,rank_in_course
0,1,Nguyen Minh Hoang,May Tinh,12,6.7,1



TOP 3 cao nhất môn: 34


,student_id,name,class,course_id,score,rank_in_course
0,2,Tran Thi Lan,Kinh Te,34,9.2,1
1,7,Bui Tien Dung,Kinh Te,34,9.2,2



TOP 3 cao nhất môn: 26


,student_id,name,class,course_id,score,rank_in_course
0,3,Pham Van Nam,Toan Tin,26,7.9,1
1,9,Duong Huu Phuc,Kinh Te,26,7.2,2
2,10,Cao Thi Hanh,May Tinh,26,7.0,3




### Kết luận:

- Kết quả trên cho biết 3 sinh viên có điểm số cao nhất trong từng môn học.  

- Việc thống kê Top 3 cao nhất theo từng môn giúp đánh giá chính xác hơn năng lực học tập của sinh viên trong từng nội dung môn học cụ thể.


3c.3 Top 3 sinh viên thấp nhất theo từng môn

In [37]:
courses = pd.read_sql("SELECT DISTINCT course_id FROM student", conn)

for mon in courses["course_id"]:
    print(f"\nTOP 3 thấp nhất môn: {mon}")

    df = pd.read_sql(f"""
    SELECT *
    FROM (
        SELECT *,
               ROW_NUMBER() OVER (PARTITION BY course_id ORDER BY score ASC) AS rank_in_course
        FROM student
    )
    WHERE course_id = {mon} AND rank_in_course <= 3
    ORDER BY rank_in_course
    """, conn)

    display(df)


TOP 3 thấp nhất môn: 12


,student_id,name,class,course_id,score,rank_in_course
0,1,Nguyen Minh Hoang,May Tinh,12,6.7,1



TOP 3 thấp nhất môn: 34


,student_id,name,class,course_id,score,rank_in_course
0,2,Tran Thi Lan,Kinh Te,34,9.2,1
1,7,Bui Tien Dung,Kinh Te,34,9.2,2



TOP 3 thấp nhất môn: 26


,student_id,name,class,course_id,score,rank_in_course
0,10,Cao Thi Hanh,May Tinh,26,7.0,1
1,9,Duong Huu Phuc,Kinh Te,26,7.2,2
2,3,Pham Van Nam,Toan Tin,26,7.9,3


### Kết luận:

- Kết quả cho biết 3 sinh viên có điểm số thấp nhất trong từng môn học.  
- Qua đó có thể xác định được những sinh viên đang gặp khó khăn ở mỗi môn, từ đó có thể đưa ra biện pháp hỗ trợ hoặc cải thiện kết quả học tập phù hợp.  

## 4.1 Thêm cột graduation_date

In [38]:
cursor.execute("""
ALTER TABLE student
ADD COLUMN graduation_date DATE
""")
conn.commit()

Nhận xét:
- Đã bổ sung thêm trường graduation_date vào bảng student
- Trường này dùng để lưu thời gian tốt nghiệp dự kiến của từng sinh viên

## 4.2 Xếp hạng sinh viên theo điểm số

In [39]:
df_rank = pd.read_sql("""
SELECT *,
       ROW_NUMBER() OVER (ORDER BY score DESC) AS rank_score
FROM student
""", conn)

display(df_rank)

,student_id,name,class,course_id,score,graduation_date,rank_score
0,2,Tran Thi Lan,Kinh Te,34,9.2,None,1
1,7,Bui Tien Dung,Kinh Te,34,9.2,None,2
2,3,Pham Van Nam,Toan Tin,26,7.9,None,3
3,9,Duong Huu Phuc,Kinh Te,26,7.2,None,4
4,10,Cao Thi Hanh,May Tinh,26,7.0,None,5
5,1,Nguyen Minh Hoang,May Tinh,12,6.7,None,6


## 4.3 Cập nhật graduation_date theo thứ hạng

In [42]:
cursor.execute("""
UPDATE student
SET graduation_date = date(
    'now',
    '+' || (
        SELECT rank_score
        FROM (
            SELECT student_id,
                   ROW_NUMBER() OVER (ORDER BY score DESC) AS rank_score
            FROM student
        ) AS ranking
        WHERE ranking.student_id = student.student_id
    ) || ' months'
)
""")
conn.commit()

### Nhận xét:
- Ngày tốt nghiệp của mỗi sinh viên được xác định dựa trên thứ hạng điểm số của sinh viên đó.  
- Sinh viên có thứ hạng cao hơn sẽ có ngày tốt nghiệp sớm hơn, còn sinh viên có thứ hạng thấp hơn sẽ có ngày tốt nghiệp muộn hơn.  


In [43]:
df = pd.read_sql("""
SELECT
    student_id,
    name,
    class,
    course_id,
    score,
    ROW_NUMBER() OVER (ORDER BY score DESC) AS rank_score,
    strftime('%d/%m/%Y', graduation_date) AS graduation_date
FROM student
ORDER BY score DESC
""", conn)

display(df)

,student_id,name,class,course_id,score,rank_score,graduation_date
0,2,Tran Thi Lan,Kinh Te,34,9.2,1,02/05/2026
1,7,Bui Tien Dung,Kinh Te,34,9.2,2,02/06/2026
2,3,Pham Van Nam,Toan Tin,26,7.9,3,02/07/2026
3,9,Duong Huu Phuc,Kinh Te,26,7.2,4,02/08/2026
4,10,Cao Thi Hanh,May Tinh,26,7.0,5,02/09/2026
5,1,Nguyen Minh Hoang,May Tinh,12,6.7,6,02/10/2026


Kết Luận:
- Kết quả cho thấy ngày tốt nghiệp của từng sinh viên đã được xác định dựa trên thứ hạng điểm số.  


- Bảng kết quả cũng thể hiện rõ mối liên hệ giữa `score`, `rank_score` và `graduation_date`, giúp việc theo dõi và đánh giá trở nên trực quan hơn.